# utils.ipynb — Fonctions utilitaires
Seed, comptage classes, résumé modèle, sauvegarde métriques.

In [ ]:
import json
import os
import random
from pathlib import Path
import numpy as np
import torch

def set_seed(seed: int = 42) -> None:
    """Fixe toutes les sources d'aléa pour la reproductibilité."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    print(f'Seed fixee a {seed}')

def count_classes(data_dir: str = 'data/chest_xray') -> dict:
    """Compte le nombre d'images par classe et par split."""
    base = Path(data_dir)
    result = {}
    for split in ('train', 'val', 'test'):
        split_path = base / split
        if not split_path.exists():
            continue
        result[split] = {}
        for class_dir in sorted(split_path.iterdir()):
            if class_dir.is_dir():
                n = len(list(class_dir.glob('*.jpeg')) +
                        list(class_dir.glob('*.jpg')) +
                        list(class_dir.glob('*.png')))
                result[split][class_dir.name] = n
    print('\nDistribution des classes :')
    for split, counts in result.items():
        total = sum(counts.values())
        parts = [f'{cls}: {n}' for cls, n in counts.items()]
        print(f'  [{split:5s}] {", ".join(parts)}  |  Total: {total}')
    return result

def model_summary(model) -> None:
    """Affiche le nombre total et entraînable de paramètres."""
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    frozen    = total - trainable
    print(f'  Parametres totaux    : {total:>12,}')
    print(f'  Parametres entraines : {trainable:>12,}')
    print(f'  Parametres geles     : {frozen:>12,}')

def save_metrics(metrics: dict, output_path: str) -> None:
    """Sauvegarde un dictionnaire de métriques au format JSON."""
    path = Path(output_path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, 'w', encoding='utf-8') as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
    print(f'Metriques sauvegardees -> {path}')

def load_metrics(output_path: str) -> dict:
    with open(output_path, 'r', encoding='utf-8') as f:
        return json.load(f)

# ── Test rapide ───────────────────────────────────────────────
set_seed(42)
print('utils.ipynb charge avec succes.')

In [ ]:
# Comptage du dataset (modifier le chemin si nécessaire)
DATA_DIR = '../data/chest_xray'
distribution = count_classes(DATA_DIR)